In [1]:
from pathlib import Path

note_files = sorted(Path("mt_onc_50").glob("*.txt"))
len(note_files), [file.name for file in note_files[:3]]

(50, ['mt_oncology_0.txt', 'mt_oncology_1.txt', 'mt_oncology_10.txt'])

In [2]:
import os
from pathlib import Path

spark_tmp = Path.home() / "spark-tmp"
spark_tmp.mkdir(parents=True, exist_ok=True)

os.environ["TMPDIR"] = str(spark_tmp)
os.environ["SPARK_LOCAL_DIRS"] = str(spark_tmp)
os.environ["JAVA_TOOL_OPTIONS"] = f"-Djava.io.tmpdir={spark_tmp}"

# Bound native inference threads on the memory-limited WSL host.
os.environ["OMP_NUM_THREADS"] = "2"
os.environ["OPENBLAS_NUM_THREADS"] = "2"

# Use the notebook interpreter for Spark workers as well.
import sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable


In [3]:
from johnsnowlabs import nlp

# Keep Spark concurrency bounded on this WSL host.
spark = nlp.start(
    master_url="local[2]",
    spark_conf={"spark.driver.memory": "3g", "spark.sql.shuffle.partitions": "2",
                "spark.default.parallelism": "2"}
)

rows = [
    (file.name, file.read_text(encoding="utf-8"))
    for file in note_files
]

notes_df = spark.createDataFrame(rows, ["file_name", "text"])

print("Number of notes in Spark:", notes_df.count())
notes_df.select("file_name").show(5, truncate=False)

📋 Loading license number 0 from /home/sengul/.johnsnowlabs/licenses/license_number_{number}_for_Spark-Healthcare.json


Picked up JAVA_TOOL_OPTIONS: -Djava.io.tmpdir=/home/sengul/spark-tmp
Picked up JAVA_TOOL_OPTIONS: -Djava.io.tmpdir=/home/sengul/spark-tmp


26/10/15 16:10:42 WARN Utils: Your hostname, Sengl resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/10/15 16:10:42 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


26/10/15 16:10:42 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


👌 Launched cpu optimized session with with: 🚀Spark-NLP==7.0.0, 💊Spark-Healthcare==7.0.0, running on ⚡ PySpark==3.5.0


Number of notes in Spark: 50


+------------------+
|file_name         |
+------------------+
|mt_oncology_0.txt |
|mt_oncology_1.txt |
|mt_oncology_10.txt|
|mt_oncology_11.txt|
|mt_oncology_12.txt|
+------------------+
only showing top 5 rows



In [4]:
sample_note = (
    notes_df
    .filter(notes_df.file_name == "mt_oncology_1.txt")
    .first()["text"]
)

print(sample_note[:2500])

Sample Type / Medical Specialty:
Hematology - Oncology
Sample Name:
BCCa Excision - Lower Lid
Description:
Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
(Medical Transcription Sample Report)
PREOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.
POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.
TITLE OF OPERATION:
Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
PROCEDURE:
The patient was brought into the operating room and prepped and draped in usual fashion. Xylocaine 2% with epinephrine was injected beneath the conjunctiva and skin of the lower lid and also beneath the conjunctiva and skin of the upper lid. A frontal nerve block was also given on the right upper lid. The anesthetic agent was also injected in th

In [5]:
document_assembler = nlp.DocumentAssembler() \
    .setInputCol("text") \
    .setOutputCol("document")

sentence_detector = nlp.SentenceDetectorDLModel.pretrained(
    "sentence_detector_dl_healthcare", "en", "clinical/models"
) \
    .setInputCols(["document"]) \
    .setOutputCol("sentence")

sentence_pipeline = nlp.Pipeline(stages=[
    document_assembler,
    sentence_detector
])

sample_note = notes_df.filter(
    notes_df.file_name == "mt_oncology_1.txt"
)

sentences_df = sentence_pipeline.fit(sample_note).transform(sample_note)

sentences_df.selectExpr(
    "file_name",
    "explode(sentence) AS sentence"
).select(
    "file_name",
    "sentence.result"
).show(10, truncate=False)

sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB
[ | ]

26/10/15 15:15:11 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


sentence_detector_dl_healthcare download started this may take some time.
Approximate size to download 367.3 KB


26/10/15 15:15:12 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


[ / ]Download done! Loading the resource.


2026-10-15 15:15:15.222305: I external/org_tensorflow/tensorflow/core/platform/cpu_feature_guard.cc:151] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 AVX512F FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


[OK!]
+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|file_name        |result                                                                                                                                                                             |
+-----------------+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|mt_oncology_1.txt|Sample Type / Medical Specialty:\nHematology - Oncology                                                                                                                            |
|mt_oncology_1.txt|Sample Name:                                                                                                                                                                   

In [6]:
sentence_rows = sentences_df.selectExpr(
    "explode(sentence) AS sentence"
).select(
    "sentence.result"
)

print("Number of text segments:", sentence_rows.count())
sentence_rows.show(100, truncate=False)

Number of text segments: 27
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|result                                                                                                                                                                                                                                                                                                                   |
+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Sample Type / Medical S

In [7]:
from pyspark.sql.functions import col

sentence_rows.filter(
    col("result").contains("Cryotherapy")
).show(truncate=False)

+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|result                                                                                                                                                                                                                           |
+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|Cryotherapy was then used to treat the nasal and temporal margins of the area of excision because of positive margins, and following this an antibiotic steroid ointment was instilled and a light pressure dressing was applied.|
+---------------------------------------------------------------------------------------

In [8]:
from johnsnowlabs import medical

tokenizer = nlp.Tokenizer() \
    .setInputCols(["sentence"]) \
    .setOutputCol("token")

word_embeddings = nlp.WordEmbeddingsModel.pretrained(
    "embeddings_clinical", "en", "clinical/models"
) \
    .setInputCols(["sentence", "token"]) \
    .setOutputCol("embeddings")

oncology_ner = medical.NerModel.pretrained(
    "ner_oncology", "en", "clinical/models"
) \
    .setInputCols(["sentence", "token", "embeddings"]) \
    .setOutputCol("ner")

ner_converter = medical.NerConverterInternal() \
    .setInputCols(["sentence", "token", "ner"]) \
    .setOutputCol("ner_chunk")

ner_pipeline = nlp.Pipeline(stages=[
    tokenizer,
    word_embeddings,
    oncology_ner,
    ner_converter
])

ner_output = ner_pipeline.fit(sentences_df).transform(sentences_df)

ner_output.selectExpr(
    "explode(ner_chunk) AS entity"
).select(
    "entity.result",
    "entity.metadata"
).show(50, truncate=False)

embeddings_clinical download started this may take some time.


26/10/15 15:15:36 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Approximate size to download 1.6 GB
[ | ]

26/10/15 15:15:37 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


embeddings_clinical download started this may take some time.
Approximate size to download 1.6 GB


...

26/10/15 15:15:52 WARN S3AbortableInputStream: Not all bytes were read from the S3ObjectInputStream, aborting HTTP connection. This is likely an error and may result in sub-optimal behavior. Request only the bytes you need via a ranged GET or drain the input stream after use.


Download done! Loading the resource.
[ — ]

2026-10-15 15:15:53.898006: W external/org_tensorflow/tensorflow/core/common_runtime/colocation_graph.cc:1218] Failed to place the graph without changing the devices of some resources. Some of the operations (that had to be colocated with resource generating operations) are not supported on the resources' devices. Current candidate devices are [
  /job:localhost/replica:0/task:0/device:CPU:0].
See below for details of this colocation group:
Colocation Debug Info:
Colocation group had the following types and supported devices: 
Root Member(assigned_device_name_index_=-1 requested_device_name_='/device:GPU:0' assigned_device_name_='' resource_device_name_='/device:GPU:0' supported_device_types_=[CPU] possible_devices_=[]
AssignSub: CPU 
ScatterAdd: CPU 
NoOp: CPU 
Sqrt: CPU 
UnsortedSegmentSum: CPU 
RealDiv: CPU 
Const: CPU 
StridedSlice: CPU 
RandomUniform: CPU 
AddV2: CPU 
Unique: CPU 
Mul: CPU 
VariableV2: CPU 
Assign: CPU 
Shape: CPU 
Identity: CPU 
Cast: CPU 
Sub: CPU 
GatherV2: CPU

[OK!]


+--------------------------+----------------------------------------------------------------------------------------------------------------+
|result                    |metadata                                                                                                        |
+--------------------------+----------------------------------------------------------------------------------------------------------------+
|Excision                  |{entity -> Cancer_Surgery, confidence -> 0.8851, ner_source -> ner_chunk, chunk -> 0, sentence -> 4}            |
|large basal cell          |{entity -> Histological_Type, confidence -> 0.7180667, ner_source -> ner_chunk, chunk -> 1, sentence -> 4}      |
|carcinoma                 |{entity -> Cancer_Dx, confidence -> 0.9946, ner_source -> ner_chunk, chunk -> 2, sentence -> 4}                 |
|right                     |{entity -> Direction, confidence -> 0.9943, ner_source -> ner_chunk, chunk -> 3, sentence -> 4}                 |
|lower

In [4]:
from sparknlp.pretrained import PretrainedPipeline

# Prefer the existing local pipeline without a model download.
cached_qa = sorted((Path.home() / "cache_pretrained").glob(
    "clinical_notes_qa_base_onnx_pipeline_en_*"
))
if cached_qa:
    qa_pipeline = PretrainedPipeline.from_disk(str(cached_qa[-1]))
else:
    qa_pipeline = PretrainedPipeline(
        "clinical_notes_qa_base_onnx_pipeline", "en", "clinical/models"
    )

question = "What was the postoperative diagnosis?"
context = sample_note.select("text").first()["text"]

qa_result = qa_pipeline.fullAnnotate([question], [context])
qa_result

Using CPUs


Using CPUs


[{'document_question': [Annotation(document, 0, 36, What was the postoperative diagnosis?, {'sentence': '0'}, [])],
  'document_context': [Annotation(document, 0, 3240, Sample Type / Medical Specialty:
   Hematology - Oncology
   Sample Name:
   BCCa Excision - Lower Lid
   Description:
   Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
   (Medical Transcription Sample Report)
   PREOPERATIVE DIAGNOSIS:
   Extremely large basal cell carcinoma, right lower lid.
   POSTOPERATIVE DIAGNOSIS:
   Extremely large basal cell carcinoma, right lower lid.
   TITLE OF OPERATION:
   Excision of large basal cell carcinoma, right lower lid, and repaired with used dorsal conjunctival flap in the upper lid and a large preauricular skin graft.
   PROCEDURE:
   The patient was brought into the operating room and prepped and draped in usual fashion. Xylocaine 2% with epinephrine was injected bene

ONNX runtime emitted 144 repeated "Closing an already closed tensor" warnings during this cell; repeated lines condensed.


In [10]:
start = context.find("POSTOPERATIVE DIAGNOSIS:")
end = context.find("TITLE OF OPERATION:", start)

evidence = context[start:end].strip()
print(evidence)

POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.


In [11]:
import re

answer_text = qa_result[0]["answer"][0].result

sections = [
    section.strip()
    for section in re.split(r"(?m)(?=^[A-Z][A-Z /-]+:\s*$)", context)
    if section.strip()
]

stop_words = {"what", "was", "the", "is", "a", "an", "of", "for", "to", "and", "in"}

query_terms = set(
    re.findall(r"[a-z0-9]+", f"{question} {answer_text}".lower())
) - stop_words

def overlap_score(section):
    section_terms = set(re.findall(r"[a-z0-9]+", section.lower()))
    return len(query_terms & section_terms)

evidence = max(sections, key=overlap_score)

print("Answer:", answer_text)
print("Evidence from the note:", evidence)

Answer: The postoperative diagnosis was extremely large basal cell carcinoma, right lower lid.
Evidence from the note: POSTOPERATIVE DIAGNOSIS:
Extremely large basal cell carcinoma, right lower lid.


In [12]:
import re
import html
from IPython.display import display, HTML

# Extract the section heading and evidence sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(r"(?<=[.!?])\s+|\n+", evidence_body)[0].strip()

# Find the evidence sentence within the correct section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

# Split the note into sentences and locate the evidence sentence
matches = list(re.finditer(r"[^.!?\n]+[.!?]?", context))
sentences = [match.group().strip() for match in matches]

center = next(
    i for i, match in enumerate(matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to three sentences before and after the evidence
start = max(0, center - 3)
end = min(len(sentences), center + 4)
nearby_sentences = sentences[start:end]

# Highlight the evidence sentence
paragraph = " ".join(
    f"<mark style='background-color:#ffe066; padding:2px'>{html.escape(sentence)}</mark>"
    if evidence_sentence in sentence
    else html.escape(sentence)
    for sentence in nearby_sentences
)

display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p><b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))

In [13]:
import re
import html
from IPython.display import display, HTML

# Get the evidence section heading and sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(r"(?<=[.!?])\s+|\n+", evidence_body)[0].strip()

# Locate the evidence sentence inside its section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

# Split the note into sentence fragments and remove standalone headings
sentence_matches = []

for match in re.finditer(r"[^.!?\n]+[.!?]?", context):
    sentence = match.group().strip()

    is_heading = re.fullmatch(r"[A-Z][A-Z0-9 /_-]*:", sentence)

    if sentence and not is_heading:
        sentence_matches.append((match, sentence))

# Find the exact occurrence selected as evidence
center = next(
    i for i, (match, _) in enumerate(sentence_matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to five sentences before and after the evidence
start = max(0, center - 5)
end = min(len(sentence_matches), center + 6)

# Highlight only the selected evidence occurrence
formatted_sentences = []

for i, (_, sentence) in enumerate(sentence_matches[start:end], start=start):
    escaped_sentence = html.escape(sentence)

    if i == center:
        escaped_sentence = (
            f"<mark style='background-color:#ffe066; padding:2px'>"
            f"{escaped_sentence}</mark>"
        )

    formatted_sentences.append(escaped_sentence)

paragraph = " ".join(formatted_sentences)

display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p style='max-width:850px; line-height:1.7'>"
    f"<b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))

In [14]:
import re
import html
from IPython.display import display, HTML

# Get the evidence section heading and sentence
heading = evidence.splitlines()[0].strip()
evidence_body = evidence.partition(":")[2].strip()
evidence_sentence = re.split(
    r"(?<=[.!?])\s+|\n+",
    evidence_body
)[0].strip()

# Find the evidence sentence within the selected section
section_start = context.find(heading)
evidence_position = context.find(evidence_sentence, section_start)

if section_start == -1 or evidence_position == -1:
    raise ValueError("Could not locate the evidence in the source note.")

# Split the source note into sentence fragments
sentence_matches = []

for match in re.finditer(r"[^.!?\n]+[.!?]?", context):
    sentence = match.group().strip()

    # Skip standalone section headings
    is_heading = re.fullmatch(r"[A-Z][A-Z0-9 /_-]*:", sentence)

    if sentence and not is_heading:
        sentence_matches.append((match, sentence))

# Find the exact sentence occurrence selected as evidence
center = next(
    i
    for i, (match, _) in enumerate(sentence_matches)
    if match.start() <= evidence_position < match.end()
)

# Include up to five sentences before and after the evidence
start = max(0, center - 5)
end = min(len(sentence_matches), center + 6)

# Highlight only the selected evidence sentence
formatted_sentences = []

for i, (_, sentence) in enumerate(sentence_matches[start:end], start=start):
    escaped_sentence = html.escape(sentence)

    if i == center:
        escaped_sentence = (
            f"<mark style='background-color:#ffe066; padding:2px'>"
            f"{escaped_sentence}</mark>"
        )

    formatted_sentences.append(escaped_sentence)

paragraph = " ".join(formatted_sentences)

# Display the answer, evidence section, and surrounding source text
display(HTML(
    f"<p><b>Answer:</b> {html.escape(answer_text)}</p>"
    f"<p><b>Evidence section:</b> {html.escape(heading)}</p>"
    f"<p style='max-width:850px; line-height:1.7'>"
    f"<b>Evidence and surrounding text:</b><br>{paragraph}</p>"
))

# Clinical Note Question Answering and Evidence Display

## Dataset Setup
- Loaded 50 oncology notes from text files into a Spark DataFrame.
- Used `mt_oncology_1.txt` as the sample clinical note for the initial workflow.

## Text Processing
- `DocumentAssembler` converted the raw text into Spark NLP document annotations.
- `SentenceDetector` split the note into sentence annotations.
- The sample note produced 27 text segments.

## Clinical Entity Extraction
- Built an NLP pipeline with a tokenizer, clinical word embeddings, the pretrained `ner_oncology` model, and `NerConverterInternal`.
- Extracted clinical entities such as cancer diagnoses, histological types, body sites, and surgical procedures.
- Filtered the results to inspect `Cancer_Dx` entities and their source sentences.

## Question Answering
- Loaded the pretrained `clinical_notes_qa_base_onnx_pipeline`.
- Asked: “What was the postoperative diagnosis?”
- The pipeline returned: “The postoperative diagnosis was extremely large basal cell carcinoma, right lower lid.”

## Evidence Display
- Selected the `POSTOPERATIVE DIAGNOSIS` section from the source note as the supporting evidence.
- Displayed the model answer separately from the source text.
- Added surrounding source sentences for context and highlighted the selected evidence sentence.

## Question-aware source evidence

The earlier cells are preserved as the original diagnosis experiment. The reusable
workflow below replaces their fixed-section evidence selection for new questions.
It reuses the loaded `qa_pipeline` and Spark DataFrame; no model download is needed.

**Evidence means source-note text offered as support for the generated answer.**
This model generates text; its answer offsets are not source-note offsets and its
score is not an evidence confidence score. We retrieve source sentences separately.
Only a unique, conservative lexical match is highlighted, with exact source offsets.
All such matches require human review: lexical agreement does not establish clinical
entailment, temporality, or that this was the model's actual reasoning. Missing,
ambiguous, paraphrased, and conflicting matches remain explicitly unverified and
have no highlighted evidence. This method deliberately favors abstention.

Planned/recommended source sentences and visibly incomplete generated answers are
also excluded conservatively. These safeguards are heuristics, not a complete
clinical temporality or factuality validator.


In [1]:
import re
import html
from collections import Counter
from IPython.display import HTML, display

TOKEN = re.compile(r"\d+(?:\.\d+)?|[a-z]+", re.I)
STOP = set("a an the was were is are be been being with of and or in on at to for by from it this that as had has have patient what which how did does do".split())
NEGATION = {"no", "not", "without", "denies", "denied", "never"}
HEADINGS = re.compile(r"(?m)^[A-Za-z][A-Za-z0-9 /_()&-]{1,79}:[ \t]*")

def terms(text):
    return [m.group().lower() for m in TOKEN.finditer(text)]

def source_sentences(text):
    """Yield exact source offsets, including notes without recognizable headings."""
    headings = list(HEADINGS.finditer(text))
    sections = [("Unsectioned text", 0, headings[0].start() if headings else len(text))]
    sections += [(m.group().strip(), m.end(),
                  headings[i + 1].start() if i + 1 < len(headings) else len(text))
                 for i, m in enumerate(headings)]
    for heading, start, end in sections:
        if heading.lower().rstrip(':') in {'keywords', 'sample name', 'sample type / medical specialty'}:
            continue
        # A decimal point is not a sentence boundary.
        for match in re.finditer(r"\S[\s\S]*?(?:[.!?](?=\s|$)|\n|$)", text[start:end]):
            raw = match.group()
            body = raw.rstrip()
            if body:
                yield heading, start + match.start(), start + match.start() + len(body)

def select_evidence(text, question, answer):
    empty = dict(evidence_status='unverified', evidence_section=None,
                 evidence_text=None, evidence_start=None, evidence_end=None,
                 surrounding_text=None, context_start=None, context_end=None)
    # Reject visibly incomplete generated text rather than supporting a partial claim.
    if any(answer.count(left) != answer.count(right) for left, right in [('(', ')'), ('[', ']')]):
        return dict(empty, evidence_reason='Generated answer appears incomplete; no evidence selected.')
    answer_terms = terms(answer)
    question_terms = set(terms(question)) - NEGATION
    required = [t for t in answer_terms if t not in STOP and t not in question_terms]
    if len(required) < 2 or re.search(r"\b(unknown|unspecified|not (?:mentioned|specified|provided)|cannot determine)\b", answer, re.I):
        return dict(empty, evidence_reason='Answer has insufficient specific source information.')
    candidates = []
    for heading, start, end in source_sentences(text):
        # Planned or recommended care is not evidence that a procedure occurred.
        # Abstain conservatively even when some words also match a completed event.
        if re.search(r"\b(will|would|should|may|might|recommend\w*|plan\w*|scheduled|consider\w*|proposed|tomorrow)\b", text[start:end], re.I):
            continue
        sentence_terms = terms(text[start:end])
        if set(answer_terms) & NEGATION != set(sentence_terms) & NEGATION:
            continue
        # Require every answer-specific token in order, retaining numbers and negation.
        iterator = iter(sentence_terms)
        if not all(any(token == needed for token in iterator) for needed in required):
            continue
        relevance = len((set(terms(heading)) - STOP) & question_terms)
        candidates.append((relevance, heading, start, end))
    if not candidates:
        return dict(empty, evidence_reason='No conservative source match; answer is unverified.')
    best_score = max(c[0] for c in candidates)
    best = [c for c in candidates if c[0] == best_score]
    if len(best) != 1:
        return dict(empty, evidence_status='ambiguous',
                    evidence_reason='Multiple source occurrences match equally; no evidence selected.')
    _, heading, start, end = best[0]
    left, right = max(0, start - 350), min(len(text), end + 350)
    return dict(evidence_status='lexical_match_review_required',
                evidence_reason='Unique lexical source match, not validated clinical entailment.',
                evidence_section=heading, evidence_text=text[start:end],
                evidence_start=start, evidence_end=end, surrounding_text=text[left:right],
                context_start=left, context_end=right)

def answer_note(file_name, text, question):
    record = dict(file_name=file_name, question=question, answer='', error=None)
    try:
        annotations = qa_pipeline.fullAnnotate([question], [text])[0].get('answer', [])
        if not annotations or not annotations[0].result.strip():
            raise ValueError('The QA pipeline returned no answer.')
        record['answer'] = annotations[0].result
        record.update(select_evidence(text, question, record['answer']))
    except Exception as exc:
        record.update(select_evidence(text, question, ''))
        record.update(evidence_status='qa_error', error=f'{type(exc).__name__}: {exc}',
                      evidence_reason='QA failed; no supporting evidence selected.')
    return record

def render_result(record):
    escape = lambda value: html.escape(str(value))
    fields = [('File', record['file_name']), ('Question', record['question']),
              ('Answer', record['answer'] or 'No answer'),
              ('Evidence status', record['evidence_status']),
              ('Selection note', record['evidence_reason']),
              ('Evidence section', record['evidence_section'] or 'Not selected')]
    content = ''.join(f'<p><b>{label}:</b> {escape(value)}</p>' for label, value in fields)
    if record['evidence_text'] is not None:
        nearby = record['surrounding_text']
        start = record['evidence_start'] - record['context_start']
        end = record['evidence_end'] - record['context_start']
        marked = (escape(nearby[:start]) + '<mark style="background:#ffe066;color:#111">'
                  + escape(nearby[start:end]) + '</mark>' + escape(nearby[end:]))
        content += '<p><b>Source-note evidence and surrounding text (review required):</b></p>'
        content += f'<div style="white-space:pre-wrap">{marked}</div>'
    else:
        content += '<p>No reliable evidence selected; no source text is highlighted.</p>'
    if record['error']:
        content += f'<p><b>Error:</b> {escape(record["error"])}</p>'
    return '<div style="max-width:950px;border-bottom:1px solid #ccc;padding:12px">' + content + '</div>'

In [2]:
# Check abstention, source offsets, section choice, and safe rendering without a model call.
test_note = "POSTOPERATIVE DIAGNOSIS:\nBasal cell carcinoma.\nPROCEDURE:\nXylocaine 2% with epinephrine was injected."
test_result = select_evidence(test_note, "What was injected?", "Xylocaine 2% with epinephrine was injected.")
assert test_result['evidence_section'] == 'PROCEDURE:'
assert test_note[test_result['evidence_start']:test_result['evidence_end']] == test_result['evidence_text']
assert select_evidence(test_note, "What was injected?", "Lidocaine 5% was injected.")['evidence_text'] is None
assert select_evidence("No aspirin was administered.", "What was administered?", "Aspirin was administered daily.")['evidence_text'] is None
assert select_evidence("Aspirin 5 mg was given. Aspirin 5 mg was given.", "What was given?", "Aspirin 5 mg.")['evidence_status'] == 'ambiguous'
assert select_evidence("Dose: 2.5 mg aspirin was given.", "What was given?", "2.5 mg aspirin.")['evidence_section'] == 'Dose:'
assert select_evidence("", "What happened?", "Unknown")['evidence_text'] is None
escaped_record = dict(test_result, file_name='<script>alert(1)</script>', question='What was injected?', answer='Xylocaine', error=None)
assert '<script>' not in render_result(escaped_record)
assert render_result(escaped_record).count('<mark ') == 1
assert select_evidence("An ultrasound of the abdomen will be performed tomorrow.", "What procedure was performed?", "An ultrasound of the abdomen was performed.")['evidence_text'] is None
assert select_evidence("He should undergo a bone marrow aspiration and biopsy.", "What procedure was performed?", "A bone marrow aspiration and biopsy was performed.")['evidence_text'] is None
assert select_evidence("Excision (0.8 cm) with reconstruction.", "What procedure was performed?", "Excision (0.8 cm) with reconstruction (")['evidence_text'] is None
print('Passed 12 evidence and rendering checks.')

Passed 12 evidence and rendering checks.


### Second question: medication used during the procedure

This asks about a different section of the same sample note. The original diagnosis
example remains above. The batch below is gated on this source-matched demonstration.

In [7]:
sample_text = notes_df.filter(notes_df.file_name == 'mt_oncology_1.txt').first()['text']
second_question = "What was injected beneath the conjunctiva and skin?"
second_result = answer_note('mt_oncology_1.txt', sample_text, second_question)
display(HTML(render_result(second_result)))
assert second_result['error'] is None, second_result['error']
assert second_result['evidence_section'] == 'PROCEDURE:', 'Review sample evidence before batching.'
assert second_result['evidence_status'] == 'lexical_match_review_required'

ONNX runtime emitted 72 repeated "Closing an already closed tensor" warnings during this cell; repeated lines condensed.


### All 50 notes

`batch_questions` is configurable. A general procedure question is used across these
heterogeneous notes; a note may not document a procedure, and generated answers may
have no support. Each file/question retains its answer, status, source section,
exact evidence offsets, surrounding text, and any error. Calls run sequentially
through the already-loaded model to bound memory. This is a 50-note demonstration,
not distributed QA inference. Long-note model truncation and paraphrases can cause
missing answers or evidence; unverified results must not be treated as established facts.

Full results are saved in [CSV](results/procedure_qa_results.csv) and an
[HTML report](results/procedure_qa_report.html). Only counts and up to three
examples are shown below. The current preview was restored from saved results
without re-executing the notebook. To export again without model inference, run
`python export_procedure_results.py` from the project directory.


In [8]:
assert second_result['evidence_status'] == 'lexical_match_review_required'
batch_questions = ["What procedure was performed?"]
batch_results = []
# Persist each completed note so a kernel interruption does not erase all answers.
import json
checkpoint_path = Path('.ipynb_checkpoints/qa_batch_results.json')
checkpoint_path.parent.mkdir(exist_ok=True)
for note_index, row in enumerate(notes_df.orderBy('file_name').toLocalIterator(), 1):
    for batch_question in batch_questions:
        batch_results.append(answer_note(row['file_name'], row['text'], batch_question))
    checkpoint_path.write_text(json.dumps(batch_results, indent=2), encoding='utf-8')
assert len(batch_results) == 50 * len(batch_questions)
assert len({(r['file_name'], r['question']) for r in batch_results}) == len(batch_results)
print('Evidence status counts:', dict(Counter(r['evidence_status'] for r in batch_results)))
print('QA error count:', sum(r['error'] is not None for r in batch_results))

Total results: 50
Evidence status counts: {'unverified': 29, 'ambiguous': 6, 'lexical_match_review_required': 15}
QA error count: 0


In [ ]:
# Export existing results and show at most three examples. No new QA calls.
from pathlib import Path
from export_procedure_results import export_results, preview_records, render_card

export_results(batch_results, Path('results'), Path('mt_onc_50'))
for record in preview_records(batch_results):
    display(HTML(render_card(record)))


Saved result: mt_oncology_12.txt (lexical_match_review_required)

Saved result: mt_oncology_1.txt (ambiguous)

Saved result: mt_oncology_0.txt (unverified)

### Validation summary — October 15, 2026

- Reused the local QA model cache; no model download was required.
- Re-executed dataset setup, the original QA call, the second question, and the 50-note QA extension. Earlier NER experiments and their outputs were retained.
- All 50 procedure questions returned model answers with no QA exceptions. Final evidence selection: 15 lexical matches requiring review, 6 ambiguous, and 29 unverified.
- All 50 results are exported to CSV and HTML in `results/`; at most three examples are displayed above. Selected evidence and surrounding text exactly match source-note character offsets. Twelve evidence/rendering assertions passed.
- Source review found planned care reported as completed care and an incomplete generated answer. Added abstention guards and reran selection/display against the saved model answers, without additional inference. These are heuristic safeguards, not comprehensive clinical validation.
- WSL became unresponsive after the initial batch and was restarted with permission. Spark concurrency is now limited, worker Python is tied to the notebook interpreter, and completed results are checkpointed after every note in the Git-ignored `.ipynb_checkpoints/qa_batch_results.json`. The checkpoint is not automatically reused.
- Repeated ONNX `Closing an already closed tensor` warnings occurred despite completed inference; their counts are retained in cell outputs.

Model interface reference: [John Snow Labs clinical notes QA pipeline](https://nlp.johnsnowlabs.com/2024/01/10/clinical_notes_qa_base_onnx_pipeline_en.html).
